# Week 12 Lab — Big Data Tools: SQL, APIs & PySpark

**Course:** Introduction to Big Data Analytics  
**Institution:** African Leadership University (AUCA)  
**Instructor:** Prince Ishimwe  
**Student Name:** Nwogu Jennifer  
**Student ID:** 27452  
**Lab:** Week 12  
**Total Marks:** 35 (+3 Bonus)

---

## Objectives

- Perform SQL queries using Python and SQLite.
- Retrieve and analyze data from APIs and JSON.
- Process large datasets using PySpark.
- Compare Pandas, SQL, Spark, and APIs for different data tasks.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import json
import requests
from pathlib import Path

print("All required libraries imported successfully!")

All required libraries imported successfully!


In [2]:
# Load the Week 12 transaction dataset
df = pd.read_csv("week12_transactions.csv")

# Display the first 5 rows
df.head()

,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


In [3]:
# Dataset overview
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nSummary statistics:")
df.describe()

Dataset shape: (2000, 7)

Column names:
['txn_id', 'date', 'hour', 'district', 'txn_type', 'amount_rwf', 'flagged']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   txn_id      2000 non-null   object
 1   date        2000 non-null   object
 2   hour        2000 non-null   int64 
 3   district    2000 non-null   object
 4   txn_type    2000 non-null   object
 5   amount_rwf  2000 non-null   int64 
 6   flagged     2000 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 109.5+ KB

Summary statistics:


,hour,amount_rwf,flagged
count,2000.000000,2000.000000,2000.000000
mean,12.910000,59938.000000,0.005500
std,5.614505,46168.432593,0.073976
min,0.000000,500.000000,0.000000
25%,9.000000,16750.000000,0.000000
50%,12.000000,55000.000000,0.000000
75%,18.000000,97000.000000,0.000000
max,23.000000,180000.000000,1.000000


In [4]:
# Create SQLite database and load transactions
conn = sqlite3.connect("momo.db")

df.to_sql("transactions", conn, if_exists="replace", index=False)

print("Database created successfully!")
print("Table 'transactions' loaded into SQLite.")

Database created successfully!
Table 'transactions' loaded into SQLite.


In [5]:
# Exercise 1.1: Largest send transactions above 100,000 RWF

query1 = """
SELECT txn_id, date, hour, district, amount_rwf
FROM transactions
WHERE txn_type = 'send'
AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5;
"""

largest_sends = pd.read_sql(query1, conn)

print("Top 5 largest send transactions above 100,000 RWF:")
display(largest_sends)

# Count flagged transactions
query2 = """
SELECT COUNT(*) AS flagged_count
FROM transactions
WHERE flagged = 1;
"""

flagged_count = pd.read_sql(query2, conn)

print("\nNumber of flagged transactions:")
display(flagged_count)

Top 5 largest send transactions above 100,000 RWF:


,txn_id,date,hour,district,amount_rwf
0,T01708,2026-11-28,15,Kicukiro,140000
1,T00364,2026-11-11,10,Huye,139000
2,T00895,2026-11-05,19,Muhanga,139000
3,T00899,2026-11-21,19,Nyarugenge,139000
4,T01314,2026-11-21,11,Gasabo,139000



Number of flagged transactions:


,flagged_count
0,11


In [6]:
# Exercise 1.2: Aggregations by transaction type

query3 = """
SELECT
    txn_type,
    COUNT(*) AS transaction_count,
    SUM(amount_rwf) AS total_amount_rwf,
    ROUND(AVG(amount_rwf), 2) AS average_amount_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_amount_rwf DESC;
"""

type_summary = pd.read_sql(query3, conn)

print("Transaction summary by type:")
display(type_summary)


# Total transaction amount per district
query4 = """
SELECT
    district,
    SUM(amount_rwf) AS total_amount_rwf
FROM transactions
GROUP BY district
ORDER BY total_amount_rwf DESC;
"""

district_summary = pd.read_sql(query4, conn)

print("\nTotal transaction amount by district:")
display(district_summary)

Transaction summary by type:


,txn_type,transaction_count,total_amount_rwf,average_amount_rwf
0,send,868,62049000,71485.02
1,paybill,330,29970000,90818.18
2,receive,489,27024000,55263.80
3,airtime,313,833000,2661.34



Total transaction amount by district:


,district,total_amount_rwf
0,Muhanga,21076000
1,Gasabo,20738000
2,Huye,18736000
3,Nyarugenge,18091000
4,Kicukiro,11338000
5,Musanze,10490500
6,Rubavu,10110500
7,Nyagatare,9296000


In [8]:
# Exercise 1.3: Compare SQL and Pandas hourly transactions

# SQL calculation
query5 = """
SELECT hour, COUNT(*) AS transaction_count
FROM transactions
GROUP BY hour
ORDER BY hour;
"""

hourly_sql = pd.read_sql(query5, conn)

# Pandas calculation
hourly_pandas = df.groupby("hour").size().reset_index(name="transaction_count")

# Verify agreement
print("Do SQL and Pandas results agree?", hourly_sql.equals(hourly_pandas))

# Show the 5 busiest hours
print("\nTop 5 busiest hours:")
display(hourly_sql.sort_values("transaction_count", ascending=False).head(5))

Do SQL and Pandas results agree? True

Top 5 busiest hours:


,hour,transaction_count
20,20,185
17,17,183
18,18,176
12,12,170
11,11,169


In [9]:
# Exercise 2.1: Parse JSON data

with open("week12_api_sample.json", "r") as file:
    api_sample = json.load(file)

print("JSON loaded successfully!")
print("Top-level keys:", api_sample.keys())

JSON loaded successfully!
Top-level keys: dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [10]:
# Exercise 2.1: Parse JSON using json.loads()

with open("week12_api_sample.json", "r") as file:
    json_string = file.read()

data = json.loads(json_string)

times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

city = "Kigali"

print("City:", city)
print("Number of readings:", len(temps))
print("Average temperature:", round(sum(temps) / len(temps), 2), "°C")

City: Kigali
Number of readings: 24
Average temperature: 21.14 °C


In [11]:
# Exercise 2.2: Fetch weather forecast using API

url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-1.9441&longitude=30.0619"
    "&hourly=temperature_2m&forecast_days=1"
    "&timezone=Africa%2FKigali"
)

try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    weather_data = response.json()
    print("Weather data fetched successfully from live API!")

except Exception as e:
    print("Live API unavailable. Using offline sample.")
    with open("week12_api_sample.json", "r") as file:
        weather_data = json.load(file)

# Extract weather data
times = weather_data["hourly"]["time"]
temps = weather_data["hourly"]["temperature_2m"]

# Calculate maximum and minimum temperatures
max_temp = max(temps)
min_temp = min(temps)

# Find the time of the hottest hour
hottest_index = temps.index(max_temp)
hottest_time = times[hottest_index]

print("\nCity: Kigali")
print("Number of hours:", len(temps))
print("Maximum temperature:", max_temp, "°C")
print("Minimum temperature:", min_temp, "°C")
print("Hottest hour:", hottest_time)

Weather data fetched successfully from live API!

City: Kigali
Number of hours: 24
Maximum temperature: 24.1 °C
Minimum temperature: 16.6 °C
Hottest hour: 2026-11-23T14:00


In [13]:
# Exercise 2.3: Weather DataFrame and Boolean Masking

weather_df = pd.DataFrame({
    "time": times,
    "temp_c": temps
})

print("First 5 weather readings:")
display(weather_df.head())

print("\nDescriptive statistics:")
display(weather_df["temp_c"].describe().round(2))

hot_hours = weather_df[weather_df["temp_c"] >= 25]

print("\nNumber of hours >= 25°C:", len(hot_hours))

First 5 weather readings:


,time,temp_c
0,2026-11-23T00:00,17.1
1,2026-11-23T01:00,16.6
2,2026-11-23T02:00,17.0
3,2026-11-23T03:00,16.7
4,2026-11-23T04:00,16.6



Descriptive statistics:


count    24.00
mean     19.05
std       2.50
min      16.60
25%      16.95
50%      18.35
75%      20.25
max      24.10
Name: temp_c, dtype: float64


Number of hours >= 25°C: 0


In [14]:
!pip install pyspark

^C


In [1]:
import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 4.3.0


In [16]:
import sys
print(sys.executable)

C:\ProgramData\anaconda3\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week12_BigDataLab") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.3.0


In [3]:
spark_df = spark.read.csv(
    "week12_transactions.csv",
    header=True,
    inferSchema=True
)

print("Total rows:", spark_df.count())
print("\nSchema:")
spark_df.printSchema()

Total rows: 2000

Schema:
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)



In [4]:
from pyspark.sql.functions import col

large_transactions = spark_df.filter(col("amount_rwf") > 100000)

print("Transactions above 100,000 RWF:", large_transactions.count())

large_transactions.show(5)

Transactions above 100,000 RWF: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00005|2026-11-15|  12|    Huye|    send|    109000|      0|
|T00009|2026-11-01|   8|  Gasabo|    send|    109000|      0|
|T00011|2026-11-22|  18|  Rubavu|    send|    102000|      0|
|T00012|2026-11-20|   9| Musanze| receive|    112000|      0|
|T00016|2026-11-03|  19| Muhanga| paybill|    154000|      0|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import sum, count, avg

type_totals = spark_df.groupBy("txn_type").agg(
    count("*").alias("transaction_count"),
    sum("amount_rwf").alias("total_amount_rwf"),
    avg("amount_rwf").alias("average_amount_rwf")
)

type_totals.orderBy("total_amount_rwf", ascending=False).show()

+--------+-----------------+----------------+------------------+
|txn_type|transaction_count|total_amount_rwf|average_amount_rwf|
+--------+-----------------+----------------+------------------+
|    send|              868|        62049000| 71485.02304147465|
| paybill|              330|        29970000| 90818.18181818182|
| receive|              489|        27024000|  55263.8036809816|
| airtime|              313|          833000|2661.3418530351437|
+--------+-----------------+----------------+------------------+



In [6]:
district_totals = spark_df.groupBy("district").agg(
    sum("amount_rwf").alias("total_amount_rwf")
)

district_totals.orderBy("total_amount_rwf", ascending=False).show()

+----------+----------------+
|  district|total_amount_rwf|
+----------+----------------+
|   Muhanga|        21076000|
|    Gasabo|        20738000|
|      Huye|        18736000|
|Nyarugenge|        18091000|
|  Kicukiro|        11338000|
|   Musanze|        10490500|
|    Rubavu|        10110500|
| Nyagatare|         9296000|
+----------+----------------+



In [7]:
spark_df.createOrReplaceTempView("transactions")

spark_sql_result = spark.sql("""
    SELECT txn_type,
           COUNT(*) AS transaction_count,
           SUM(amount_rwf) AS total_amount_rwf
    FROM transactions
    GROUP BY txn_type
    ORDER BY total_amount_rwf DESC
""")

spark_sql_result.show()

+--------+-----------------+----------------+
|txn_type|transaction_count|total_amount_rwf|
+--------+-----------------+----------------+
|    send|              868|        62049000|
| paybill|              330|        29970000|
| receive|              489|        27024000|
| airtime|              313|          833000|
+--------+-----------------+----------------+



1. SQL — I would use SQL because the Ministry already provides access to a PostgreSQL database, making it efficient to calculate monthly totals grouped by district.

2. Pandas — I would use Pandas because it is suitable for cleaning a 40,000-row CSV, preparing data for charts, and working with data for regression modeling.

3. Spark — I would use Spark because it is designed for distributed processing of very large datasets such as 40 TB across a cluster.

4. API — I would use an API because it allows the dashboard to retrieve current weather data automatically without manually uploading files.